<a href="https://colab.research.google.com/github/VijayaRagav07/MMB/blob/main/Benchmark_7_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys


# ==========================================================
# BLOCK 1 : IMPORTS & ENVIRONMENT
# ==========================================================

# ==========================
# Standard Libraries
# ==========================

import os
import gc
import time
import random
import warnings

warnings.filterwarnings("ignore")

# ==========================
# Numerical Libraries
# ==========================

import numpy as np
import pandas as pd

# ==========================
# Visualization
# ==========================

import matplotlib.pyplot as plt
import seaborn as sns

# ==========================
# TensorFlow
# ==========================

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras import mixed_precision

from tensorflow.keras.callbacks import (

    EarlyStopping,

    ReduceLROnPlateau,

    ModelCheckpoint

)

# ==========================
# Evaluation Metrics
# ==========================

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    roc_auc_score,

    roc_curve,

    precision_recall_curve,

    auc,

    matthews_corrcoef,

    cohen_kappa_score

)

# ==========================================================
# RANDOM SEED
# ==========================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

tf.random.set_seed(SEED)

# ==========================================================
# GPU OPTIMIZATION
# ==========================================================

gpus = tf.config.list_physical_devices('GPU')

if gpus:

    try:

        for gpu in gpus:

            tf.config.experimental.set_memory_growth(gpu, True)

        print(f"GPU Available : {len(gpus)}")

    except RuntimeError as e:

        print(e)

else:

    print("Running on CPU")

# ==========================================================
# MIXED PRECISION
# Saves GPU memory
# ==========================================================

mixed_precision.set_global_policy("mixed_float16")

print("Precision Policy :", mixed_precision.global_policy())

# ==========================================================
# XLA
# ==========================================================

tf.config.optimizer.set_jit(True)

# ==========================================================
# OUTPUT DIRECTORY
# ==========================================================

OUTPUT_DIR = "/content/Benchmark_Results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

os.makedirs(f"{OUTPUT_DIR}/weights", exist_ok=True)

os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)

os.makedirs(f"{OUTPUT_DIR}/metrics", exist_ok=True)

# ==========================================================
# MEMORY CLEANER
# ==========================================================

def clear_memory():

    tf.keras.backend.clear_session()

    gc.collect()

# ==========================================================
# SHOW VERSION
# ==========================================================

print("="*60)

print("TensorFlow :", tf.__version__)

print("="*60)

Running on CPU
Precision Policy : <DTypePolicy "mixed_float16">
TensorFlow : 2.20.0


In [2]:
# ==========================================================
# BLOCK 2 : CONFIGURATION
# ==========================================================

from google.colab import drive

# ==========================================================
# MOUNT GOOGLE DRIVE
# ==========================================================

drive.mount('/content/drive', force_remount=True)

# ==========================================================
# DATASET LOCATION
# ==========================================================

BASE_PATH = "/content/drive/MyDrive/MMB datsets"

DATASETS = {

    "Ultrasound": os.path.join(BASE_PATH, "Ultrasound Images_MSI"),

    "Histopathology": os.path.join(BASE_PATH, "Histopathological_MSI"),

    "ChestXray": os.path.join(BASE_PATH, "Chest_XRay_MSI")

}

# ==========================================================
# VERIFY DATASETS
# ==========================================================

print("="*60)

for name, path in DATASETS.items():

    if os.path.exists(path):
        print(f"{name:<18} ✓ Found")
    else:
        raise FileNotFoundError(f"{path} not found")

print("="*60)

# ==========================================================
# MODELS TO BENCHMARK
# ==========================================================

MODELS = [

    "ResNet50",

    "DenseNet121",

    "EfficientNetB0",

    "Xception",

    "InceptionV3",

    "MobileNetV3Large",

    "ConvNeXtTiny"

]

# ==========================================================
# IMAGE SIZE
# ==========================================================

IMAGE_SIZE = {

    "ResNet50": (224,224),

    "DenseNet121": (224,224),

    "EfficientNetB0": (224,224),

    "MobileNetV3Large": (224,224),

    "ConvNeXtTiny": (224,224),

    "Xception": (299,299),

    "InceptionV3": (299,299)

}

# ==========================================================
# TRAINING PARAMETERS
# ==========================================================

BATCH_SIZE = 16

EPOCHS = 20

LEARNING_RATE = 1e-4

NUM_CLASSES = 2

# ==========================================================
# ALL CROSS-MODALITY EXPERIMENTS
# ==========================================================

EXPERIMENTS = [

    ("Ultrasound","Histopathology"),

    ("Ultrasound","ChestXray"),

    ("Histopathology","Ultrasound"),

    ("Histopathology","ChestXray"),

    ("ChestXray","Ultrasound"),

    ("ChestXray","Histopathology")

]

print("\nCross-Modality Experiments\n")

for train_modality, test_modality in EXPERIMENTS:

    print(f"Train : {train_modality:15} ---> Test : {test_modality}")

# ==========================================================
# RESULTS STORAGE
# ==========================================================

benchmark_results = []

# ==========================================================
# OUTPUT FILES
# ==========================================================

RESULTS_CSV = os.path.join(

    OUTPUT_DIR,

    "metrics",

    "Benchmark_Results.csv"

)

print("\nConfiguration Loaded Successfully.")

Mounted at /content/drive
Ultrasound         ✓ Found
Histopathology     ✓ Found
ChestXray          ✓ Found

Cross-Modality Experiments

Train : Ultrasound      ---> Test : Histopathology
Train : Ultrasound      ---> Test : ChestXray
Train : Histopathology  ---> Test : Ultrasound
Train : Histopathology  ---> Test : ChestXray
Train : ChestXray       ---> Test : Ultrasound
Train : ChestXray       ---> Test : Histopathology

Configuration Loaded Successfully.


In [3]:
# ==========================================================
# BLOCK 3 : DATA LOADER
# ==========================================================

AUTOTUNE = tf.data.AUTOTUNE

# ----------------------------------------------------------
# CLASS LABELS
# ----------------------------------------------------------

CLASS_NAMES = sorted(
    os.listdir(DATASETS["Ultrasound"])
)

CLASS_TO_INDEX = {

    name: idx

    for idx, name in enumerate(CLASS_NAMES)

}

print("Classes :", CLASS_TO_INDEX)


# ----------------------------------------------------------
# GET IMAGE PATHS
# ----------------------------------------------------------

def get_image_paths(dataset_path):

    image_paths = []

    labels = []

    for class_name in CLASS_NAMES:

        class_dir = os.path.join(dataset_path, class_name)

        if not os.path.exists(class_dir):
            continue

        for image_name in os.listdir(class_dir):

            if image_name.lower().endswith(
                (".jpg",".jpeg",".png",".bmp",".tif",".tiff")
            ):

                image_paths.append(

                    os.path.join(class_dir,image_name)

                )

                labels.append(

                    CLASS_TO_INDEX[class_name]

                )

    return image_paths, labels


# ----------------------------------------------------------
# IMAGE LOADER
# ----------------------------------------------------------

def load_image(path,label,image_size):

    image = tf.io.read_file(path)

    image = tf.image.decode_image(

        image,

        channels=3,

        expand_animations=False

    )

    image = tf.image.resize(

        image,

        image_size

    )

    image = tf.cast(image,tf.float32)

    return image,label


# ----------------------------------------------------------
# BUILD DATASET
# ----------------------------------------------------------

def create_dataset(

    dataset_path,

    image_size,

    shuffle=True

):

    image_paths,labels = get_image_paths(dataset_path)

    ds = tf.data.Dataset.from_tensor_slices(

        (image_paths,labels)

    )

    if shuffle:

        ds = ds.shuffle(

            len(image_paths),

            seed=SEED,

            reshuffle_each_iteration=True

        )

    ds = ds.map(

        lambda x,y:

        load_image(

            x,

            y,

            image_size

        ),

        num_parallel_calls=AUTOTUNE

    )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(AUTOTUNE)

    return ds


print("Dataset Loader Ready.")

Classes : {'Benign': 0, 'Malignant': 1}
Dataset Loader Ready.


In [4]:
from tensorflow.keras import regularizers

# ----------------------------------------------------------
# IMPORT PRETRAINED MODELS
# ----------------------------------------------------------

from tensorflow.keras.applications import (

    ResNet50,

    DenseNet121,

    EfficientNetB0,

    Xception,

    InceptionV3,

    MobileNetV3Large,

    ConvNeXtTiny

)

# ----------------------------------------------------------
# IMPORT PREPROCESSING FUNCTIONS
# ----------------------------------------------------------

from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess

from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenet_preprocess

from tensorflow.keras.applications.convnext import preprocess_input as convnext_preprocess


# ----------------------------------------------------------
# BACKBONES
# ----------------------------------------------------------

BACKBONES = {

    "ResNet50": ResNet50,

    "DenseNet121": DenseNet121,

    "EfficientNetB0": EfficientNetB0,

    "Xception": Xception,

    "InceptionV3": InceptionV3,

    "MobileNetV3Large": MobileNetV3Large,

    "ConvNeXtTiny": ConvNeXtTiny

}


# ----------------------------------------------------------
# PREPROCESS FUNCTIONS
# ----------------------------------------------------------

PREPROCESS = {

    "ResNet50": resnet_preprocess,

    "DenseNet121": densenet_preprocess,

    "EfficientNetB0": efficientnet_preprocess,

    "Xception": xception_preprocess,

    "InceptionV3": inception_preprocess,

    "MobileNetV3Large": mobilenet_preprocess,

    "ConvNeXtTiny": convnext_preprocess

}


# ----------------------------------------------------------
# BUILD MODEL
# ----------------------------------------------------------

def build_model(model_name):

    image_size = IMAGE_SIZE[model_name]

    preprocess = PREPROCESS[model_name]

    # ----------------------------------------------------------
    # DATA AUGMENTATION (Moved inside function)
    # ----------------------------------------------------------

    data_augmentation = tf.keras.Sequential([

        layers.RandomFlip("horizontal"),

        layers.RandomRotation(0.10),

        layers.RandomZoom(0.10),

        layers.RandomContrast(0.10)

    ])

    backbone = BACKBONES[model_name](

        include_top=False,

        weights="imagenet",

        input_shape=(*image_size,3)

    )

    backbone.trainable = False

    inputs = tf.keras.Input(

        shape=(*image_size,3)

    )

    x = data_augmentation(inputs)

    x = preprocess(x)

    x = backbone(

        x,

        training=False

    )

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.40)(x)

    x = layers.Dense(

        512,

        activation="relu",

        kernel_regularizer=regularizers.l2(1e-4)

    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.30)(x)

    outputs = layers.Dense(

        1,

        activation="sigmoid",

        dtype="float32"

    )(x)

    model = tf.keras.Model(

        inputs,

        outputs,

        name=model_name

    )

    model.compile(

        optimizer=tf.keras.optimizers.Adam(

            learning_rate=LEARNING_RATE

        ),

        loss="binary_crossentropy",

        metrics=[

            tf.keras.metrics.BinaryAccuracy(

                name="accuracy"

            ),

            tf.keras.metrics.Precision(

                name="precision"

            ),

            tf.keras.metrics.Recall(

                name="recall"

            ),

            tf.keras.metrics.AUC(

                name="auc"

            )

        ]

    )

    return model

In [8]:
# ==========================================================
# BLOCK 6 : OPTIMIZED TRAINING PIPELINE
# ==========================================================

import os
import time

# ----------------------------------------------------------
# CALLBACKS
# ----------------------------------------------------------

def get_callbacks(model_name, train_modality):

    model_path = os.path.join(
        OUTPUT_DIR,
        "weights",
        f"{model_name}_{train_modality}.keras"
    )

    callbacks = [

        EarlyStopping(
            monitor="loss",
            patience=3,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="loss",
            factor=0.2,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=model_path,
            monitor="loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=0
        )

    ]

    return callbacks


# ----------------------------------------------------------
# TRAIN MODEL
# ----------------------------------------------------------

def train_model(
    model,
    train_dataset,
    model_name,
    train_modality
):

    print("\n" + "=" * 70)
    print(f"Training Model : {model_name}")
    print(f"Training Data  : {train_modality}")
    print("=" * 70)

    start_time = time.time()

    history = model.fit(

        train_dataset,

        epochs=EPOCHS,

        callbacks=get_callbacks(
            model_name,
            train_modality
        ),

        verbose=1

    )

    training_time = time.time() - start_time

    return model, history, training_time


# ----------------------------------------------------------
# PARAMETER COUNT
# ----------------------------------------------------------

def get_parameter_count(model):

    trainable = np.sum(
        [np.prod(v.shape) for v in model.trainable_weights]
    )

    frozen = np.sum(
        [np.prod(v.shape) for v in model.non_trainable_weights]
    )

    total = trainable + frozen

    return {

        "Total": int(total),

        "Trainable": int(trainable),

        "Frozen": int(frozen)

    }


# ----------------------------------------------------------
# MEMORY CLEANUP
# ----------------------------------------------------------

def cleanup():

    tf.keras.backend.clear_session()

    gc.collect()

In [9]:
# ==========================================================
# BLOCK 7 : OPTIMIZED MODEL EVALUATION
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    matthews_corrcoef,
    cohen_kappa_score,
    auc
)

import numpy as np
import time


# ----------------------------------------------------------
# SPECIFICITY
# ----------------------------------------------------------

def specificity_score(cm):

    if cm.shape != (2,2):
        return 0.0

    tn, fp, fn, tp = cm.ravel()

    return tn / (tn + fp + 1e-8)


# ----------------------------------------------------------
# EVALUATION
# ----------------------------------------------------------

def evaluate_model(
        model,
        test_dataset,
        model_name,
        train_modality,
        test_modality,
        training_time
):

    print(f"\nEvaluating {model_name}")

    # ---------------------------------------
    # True Labels
    # ---------------------------------------

    y_true = np.concatenate(

        [labels.numpy() for _, labels in test_dataset]

    ).astype(np.int32)

    # ---------------------------------------
    # Prediction
    # ---------------------------------------

    start = time.time()

    y_prob = model.predict(
        test_dataset,
        verbose=0
    ).ravel()

    inference_time = time.time() - start

    y_pred = (y_prob >= 0.5).astype(np.int32)

    # ---------------------------------------
    # Confusion Matrix
    # ---------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0,1]
    )

    # ---------------------------------------
    # Metrics
    # ---------------------------------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    specificity = specificity_score(cm)

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except:
        roc_auc = np.nan

    try:

        p, r, _ = precision_recall_curve(
            y_true,
            y_prob
        )

        pr_auc = auc(r, p)

    except:

        pr_auc = np.nan

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    kappa = cohen_kappa_score(
        y_true,
        y_pred
    )

    params = get_parameter_count(model)

    # ---------------------------------------
    # Return
    # ---------------------------------------

    results = {

        "Model": model_name,

        "Train Modality": train_modality,

        "Test Modality": test_modality,

        "Accuracy": round(accuracy,4),

        "Precision": round(precision,4),

        "Recall": round(recall,4),

        "Specificity": round(specificity,4),

        "F1 Score": round(f1,4),

        "ROC AUC": round(roc_auc,4) if not np.isnan(roc_auc) else np.nan,

        "PR AUC": round(pr_auc,4) if not np.isnan(pr_auc) else np.nan,

        "MCC": round(mcc,4),

        "Kappa": round(kappa,4),

        "Training Time (s)": round(training_time,2),

        "Inference Time (s)": round(inference_time,2),

        "Parameters": params["Total"],

        "Trainable": params["Trainable"],

        "Frozen": params["Frozen"]

    }

    return results, cm, y_true, y_pred, y_prob

In [10]:
# ==========================================================
# BLOCK 6 : MODEL EVALUATION
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    matthews_corrcoef,
    cohen_kappa_score,
    auc
)

# ----------------------------------------------------------
# SPECIFICITY
# ----------------------------------------------------------

def specificity_score(y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()

    return tn / (tn + fp)


# ----------------------------------------------------------
# EVALUATE MODEL
# ----------------------------------------------------------

def evaluate_model(

    model,

    test_dataset,

    model_name,

    train_modality,

    test_modality,

    training_time

):

    print("\nEvaluating...")

    # ------------------------------------
    # Inference
    # ------------------------------------

    start = time.time()

    probabilities = model.predict(

        test_dataset,

        verbose=0

    )

    inference_time = time.time() - start

    probabilities = probabilities.flatten()

    predictions = (probabilities >= 0.5).astype(np.int32)

    # ------------------------------------
    # True labels
    # ------------------------------------

    y_true = np.concatenate(

        [y.numpy() for _, y in test_dataset]

    ).astype(np.int32)

    # ------------------------------------
    # Metrics
    # ------------------------------------

    accuracy = accuracy_score(

        y_true,

        predictions

    )

    precision = precision_score(

        y_true,

        predictions,

        zero_division=0

    )

    recall = recall_score(

        y_true,

        predictions,

        zero_division=0

    )

    specificity = specificity_score(

        y_true,

        predictions

    )

    f1 = f1_score(

        y_true,

        predictions,

        zero_division=0

    )

    roc_auc = roc_auc_score(

        y_true,

        probabilities

    )

    precision_curve, recall_curve, _ = precision_recall_curve(

        y_true,

        probabilities

    )

    pr_auc = auc(

        recall_curve,

        precision_curve

    )

    mcc = matthews_corrcoef(

        y_true,

        predictions

    )

    kappa = cohen_kappa_score(

        y_true,

        predictions

    )

    cm = confusion_matrix(

        y_true,

        predictions

    )

    params = get_parameter_count(model)

    # ------------------------------------
    # STORE RESULTS
    # ------------------------------------

    results = {

        "Model": model_name,

        "Train Modality": train_modality,

        "Test Modality": test_modality,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "Specificity": specificity,

        "F1 Score": f1,

        "ROC AUC": roc_auc,

        "PR AUC": pr_auc,

        "MCC": mcc,

        "Kappa": kappa,

        "Training Time": training_time,

        "Inference Time": inference_time,

        "Parameters": params["Total"],

        "Trainable": params["Trainable"],

        "Frozen": params["Frozen"],

        "Confusion Matrix": cm

    }

    return results

In [11]:
# ==========================================================
# BLOCK 8 : VISUALIZATION FUNCTIONS
# ==========================================================

import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


# ----------------------------------------------------------
# TRAINING CURVES
# ----------------------------------------------------------

def plot_training_history(history, model_name):

    fig, ax = plt.subplots(1, 2, figsize=(12,5))

    # Accuracy
    ax[0].plot(history.history["accuracy"], linewidth=2)
    ax[0].set_title(f"{model_name} Accuracy")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Accuracy")

    # Loss
    ax[1].plot(history.history["loss"], linewidth=2)
    ax[1].set_title(f"{model_name} Loss")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Loss")

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            OUTPUT_DIR,
            "plots",
            f"{model_name}_training.png"
        ),

        dpi=300,
        bbox_inches="tight"

    )

    plt.close()


# ----------------------------------------------------------
# CONFUSION MATRIX
# ----------------------------------------------------------

def plot_confusion_matrix(cm, model_name,
                          train_modality,
                          test_modality):

    plt.figure(figsize=(6,5))

    sns.heatmap(

        cm,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=CLASS_NAMES,

        yticklabels=CLASS_NAMES

    )

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.title(

        f"{model_name}\n"

        f"{train_modality} → {test_modality}"

    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(

            OUTPUT_DIR,

            "plots",

            f"{model_name}_{train_modality}_{test_modality}_CM.png"

        ),

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()


# ----------------------------------------------------------
# ROC CURVE
# ----------------------------------------------------------

def plot_roc_curve(

    y_true,

    probabilities,

    model_name,

    train_modality,

    test_modality

):

    fpr, tpr, _ = roc_curve(

        y_true,

        probabilities

    )

    roc_auc = auc(fpr,tpr)

    plt.figure(figsize=(6,6))

    plt.plot(

        fpr,

        tpr,

        linewidth=2,

        label=f"AUC = {roc_auc:.4f}"

    )

    plt.plot([0,1],[0,1],'k--')

    plt.xlabel("False Positive Rate")

    plt.ylabel("True Positive Rate")

    plt.title(

        f"{model_name}\n"

        f"{train_modality} → {test_modality}"

    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(

        os.path.join(

            OUTPUT_DIR,

            "plots",

            f"{model_name}_{train_modality}_{test_modality}_ROC.png"

        ),

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()


# ----------------------------------------------------------
# PRECISION-RECALL CURVE
# ----------------------------------------------------------

def plot_pr_curve(

    y_true,

    probabilities,

    model_name,

    train_modality,

    test_modality

):

    precision, recall, _ = precision_recall_curve(

        y_true,

        probabilities

    )

    pr_auc = auc(recall, precision)

    plt.figure(figsize=(6,6))

    plt.plot(

        recall,

        precision,

        linewidth=2,

        label=f"PR AUC = {pr_auc:.4f}"

    )

    plt.xlabel("Recall")

    plt.ylabel("Precision")

    plt.title(

        f"{model_name}\n"

        f"{train_modality} → {test_modality}"

    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(

        os.path.join(

            OUTPUT_DIR,

            "plots",

            f"{model_name}_{train_modality}_{test_modality}_PR.png"

        ),

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()


print("Visualization Functions Loaded Successfully.")

Visualization Functions Loaded Successfully.


In [12]:
# ==========================================================
# BLOCK 9 : FINAL BENCHMARK REPORT
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------
# CONVERT RESULTS TO DATAFRAME
# ----------------------------------------------------------

results_df = pd.DataFrame(benchmark_results)

# Remove confusion matrix before saving
save_df = results_df.drop(columns=["Confusion Matrix"])

# Save CSV
save_df.to_csv(

    RESULTS_CSV,

    index=False

)

print(f"\nResults saved to:\n{RESULTS_CSV}")

# ----------------------------------------------------------
# AVERAGE PERFORMANCE OF EACH MODEL
# ----------------------------------------------------------

summary_df = save_df.groupby("Model").agg({

    "Accuracy":"mean",

    "Precision":"mean",

    "Recall":"mean",

    "Specificity":"mean",

    "F1 Score":"mean",

    "ROC AUC":"mean",

    "PR AUC":"mean",

    "MCC":"mean",

    "Kappa":"mean",

    "Training Time":"mean",

    "Inference Time":"mean"

}).round(4)

summary_df = summary_df.sort_values(

    by="Accuracy",

    ascending=False

)

summary_path = os.path.join(

    OUTPUT_DIR,

    "metrics",

    "Average_Model_Performance.csv"

)

summary_df.to_csv(summary_path)

print("\nAverage performance saved.")

print(summary_df)

# ----------------------------------------------------------
# BEST MODEL
# ----------------------------------------------------------

best_model = summary_df.index[0]

print("\n"+"="*60)

print("BEST PRETRAINED CNN")

print("="*60)

print(best_model)

print("="*60)

# ----------------------------------------------------------
# BAR CHARTS
# ----------------------------------------------------------

metrics = [

    "Accuracy",

    "F1 Score",

    "ROC AUC",

    "MCC"

]

for metric in metrics:

    plt.figure(figsize=(10,5))

    summary_df[metric].sort_values().plot(

        kind="barh"

    )

    plt.title(metric)

    plt.tight_layout()

    plt.savefig(

        os.path.join(

            OUTPUT_DIR,

            "plots",

            f"{metric}.png"

        ),

        dpi=300

    )

    plt.close()

# ----------------------------------------------------------
# HEATMAP
# ----------------------------------------------------------

plt.figure(figsize=(12,6))

sns.heatmap(

    summary_df[

        [

            "Accuracy",

            "Precision",

            "Recall",

            "Specificity",

            "F1 Score",

            "ROC AUC",

            "MCC"

        ]

    ],

    annot=True,

    cmap="viridis",

    fmt=".3f"

)

plt.title("CNN Benchmark Summary")

plt.tight_layout()

plt.savefig(

    os.path.join(

        OUTPUT_DIR,

        "plots",

        "Benchmark_Heatmap.png"

    ),

    dpi=300

)

plt.close()

print("\nHeatmap saved.")

# ----------------------------------------------------------
# RANKING
# ----------------------------------------------------------

ranking = summary_df.reset_index()

ranking["Rank"] = range(

    1,

    len(ranking)+1

)

ranking = ranking[

    [

        "Rank",

        "Model",

        "Accuracy",

        "F1 Score",

        "ROC AUC",

        "MCC"

    ]

]

ranking_path = os.path.join(

    OUTPUT_DIR,

    "metrics",

    "Model_Ranking.csv"

)

ranking.to_csv(

    ranking_path,

    index=False

)

print("\nRanking Saved.")

print(ranking)

print("\nBenchmark Completed Successfully.")

KeyError: "['Confusion Matrix'] not found in axis"